# **Segmentación y vectorización de vías a partir de imágenes GeoTIFF**

Este script contiene funciones para procesar imágenes satelitales georreferenciadas y realizar segmentación de vías mediante técnicas de procesamiento digital de imágenes. El resultado es una máscara binaria exportada como GeoTIFF y una conversión opcional de áreas específicas del raster a formato vectorial (Shapefile).

> ⚠️ **Nota importante**: Este método está diseñado para funcionar específicamente con imágenes descargadas desde servicios de *Tile Map Service* (TMS) del tipo **"Terrain"**, ya que este tipo de fondo resalta los caminos y vías con alto contraste, facilitando la segmentación automática.

### ✅ Funcionalidades principales:

1. **`analyze_and_segment_roads`**:
   - Aplica umbralización de Otsu sobre una imagen `GeoTIFF` (tipo *Terrain*) para identificar posibles áreas de vía.
   - Realiza operaciones morfológicas (`opening` y `closing`) para reducir ruido y suavizar los bordes detectados.
   - Filtra objetos pequeños utilizando un umbral de área mínima (`min_area`).
   - Exporta la máscara binaria como archivo `.tif` georreferenciado.

2. **`raster_to_shapefile`**:
   - Convierte las regiones del raster con un valor específico (por defecto `0`) en polígonos vectoriales.
   - Extrae las geometrías mediante `rasterio.features.shapes`.
   - Guarda las geometrías como archivo `Shapefile`, listo para ser utilizado en plataformas SIG como QGIS o ArcGIS.

### 🧪 Casos de uso:
- Análisis y seguimiento de la red vial a partir de imágenes públicas.
- Preprocesamiento de datasets para entrenamiento de modelos de detección de carreteras.
- Generación automática de capas vectoriales a partir de fondos tipo terreno.

### 🛠️ Requisitos clave:
- La imagen de entrada debe ser un **GeoTIFF georreferenciado** proveniente de un fondo tipo **Terrain**.
- Las funciones utilizan librerías como `rasterio`, `scikit-image`, `shapely`, `geopandas`, y `matplotlib`.


In [ ]:
#@title Funciones
import rasterio
from rasterio.plot import show
from skimage.filters import threshold_otsu
from skimage.morphology import binary_opening, binary_closing, disk, remove_small_objects
from skimage.measure import label, regionprops
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import box, shape
import numpy as np
from rasterio.features import shapes


def analyze_and_segment_roads(image_path, output_raster, min_area=100):
    """
    Perform road segmentation on a georeferenced image.

    Parameters:
        image_path (str): Path to the input GeoTIFF image.
        output_raster (str): Path to save the binary mask as a GeoTIFF.
        min_area (int): Minimum area of connected components to be considered a road.

    Returns:
        None
    """
    with rasterio.open(image_path) as src:
        # Read the first band
        band = src.read(1)

        # Apply Otsu's thresholding
        thresh = threshold_otsu(band)
        binary_mask = band > thresh

        # Perform morphological operations
        refined_mask = binary_closing(binary_opening(binary_mask, disk(3)), disk(3))

        # Remove small objects and noise
        refined_mask = remove_small_objects(refined_mask, min_size=min_area)

        # Save the refined mask as a GeoTIFF
        with rasterio.open(
            output_raster,
            "w",
            driver="GTiff",
            height=refined_mask.shape[0],
            width=refined_mask.shape[1],
            count=1,
            dtype=rasterio.uint8,  # Corrected dtype
            crs=src.crs,
            transform=src.transform,
        ) as dst:
            dst.write((refined_mask * 255).astype(rasterio.uint8), 1)  # Scale boolean to uint8


# Función para convertir el color 0 en un shapefile
def raster_to_shapefile(input_raster_path, output_shapefile_path, target_value=0):
    # Abrir el raster
    with rasterio.open(input_raster_path) as src:
        raster_data = src.read(1)  # Leer la primera banda
        raster_transform = src.transform  # Obtener la transformación del raster

        # Crear una máscara donde los valores coinciden con el target_value (0 en este caso)
        mask = raster_data == target_value

        # Extraer las geometrías (polígonos) usando rasterio.features.shapes
        shapes_generator = shapes(raster_data, mask=mask, transform=raster_transform)

        # Convertir las geometrías a un formato compatible con GeoPandas
        geometries = []
        for geom, value in shapes_generator:
            if value == target_value:  # Solo incluir geometrías con el valor objetivo
                geometries.append(shape(geom))

    # Crear un GeoDataFrame
    gdf = gpd.GeoDataFrame(geometry=geometries, crs=src.crs)

    # Guardar el GeoDataFrame como shapefile
    gdf.to_file(output_shapefile_path)


In [ ]:
import os, glob

troncal='Troncal9'
root='./data'#Reemplazar con carpeta root con las imagenes
z='19'
BB='22'

#Carpeta con imágenes tipo Terrain
terr='SourceTerrain_Z{}_SegmentMts100_ExtBB{}'.format(z,BB)#Reemplazar con el nombre de la carpeta que contiene las imágenes tipo terrain
terr_path=os.path.join(root,terr,troncal)

#Listado de imágenes
list_terr=sorted(glob.glob(os.path.join(terr_path,'*.tif')))

print('Terrain: ',len(list_terr))

In [ ]:
import os
from tqdm import tqdm

# Configurar carpeta de máscaras
mask_fol = os.path.join(root.replace('IMAGES', 'MASKS_Z{}_BB{}'.format(z, BB)), troncal)
os.makedirs(mask_fol, exist_ok=True)

# Iterar sobre las imágenes
for image_path in tqdm(list_terr):
    # Ruta del archivo de salida del raster
    output_raster = os.path.join(mask_fol, os.path.basename(image_path))
    
    # Verificar si el raster de salida ya existe
    if not os.path.exists(output_raster):
        # Si no existe, procesar la imagen
        analyze_and_segment_roads(image_path, output_raster)
        
        # Generar y guardar el shapefile si el raster fue creado
        output_shapefile = output_raster.replace('.tif', '.shp')
        raster_to_shapefile(output_raster, output_shapefile)
    else:
        # Si el archivo ya existe, saltar al siguiente
        print(f"El archivo ya existe, saltando: {output_raster}")


#VISUALIZAR UNO DE LOS RESULTADOS

import leafmap
list_shp=glob.glob(os.path.join(mask_fol,'*.shp'))
m=leafmap.Map()
m.add_vector(list_shp[0])
m

100%|██████████| 6684/6684 [32:29<00:00,  3.43it/s]  


# **Postpocesamiento 1. Filtrado de máscaras vectoriales para extraer únicamente vías reales**

Este script realiza un paso de **postprocesamiento** sobre los shapefiles generados durante la segmentación de imágenes tipo *Terrain*. Debido a que en dicho proceso pueden detectarse elementos irrelevantes (como señales, etiquetas o bordes decorativos del mapa), se aplica un filtro por área para conservar únicamente las geometrías que probablemente corresponden a **vías reales**.

### ✅ Funcionalidades principales:

- 📂 **Carga automática de shapefiles** generados previamente en la carpeta `MASKS`.
- ⚙️ **Filtrado por área**: descarta polígonos cuya superficie sea menor a un umbral (`area_minima`, configurable por el usuario).
- 💾 **Exportación dual**: guarda el resultado filtrado tanto en formato `Shapefile` como en `GeoJSON` para su uso en distintos entornos SIG.
- 🧠 **Evita reprocesar archivos existentes**: si el archivo filtrado ya existe, se omite automáticamente.

### 🎯 Justificación del proceso:

Durante la segmentación inicial basada en imágenes del tipo **Terrain**, se pueden detectar objetos pequeños que no son vías (por ejemplo, números de elevación, líneas finas o marcadores del mapa base). Este paso permite limpiar el conjunto de datos y **preservar únicamente las formas relevantes para el análisis de infraestructura vial**.

### 🛠️ Parámetros configurables:

- `area_minima`: área mínima (en unidades del sistema de referencia del shapefile) que debe tener una geometría para ser conservada.
- `mask_fol`: carpeta que contiene los shapefiles sin filtrar (`MASKS`).
- `vias_fol`: carpeta de salida donde se guardarán los shapefiles filtrados (`MASKS_VIAS`).

Este paso mejora significativamente la calidad de los datos vectoriales generados a partir de imágenes ráster, facilitando su uso posterior en análisis, visualización y entrenamiento de modelos.


In [16]:
#FILTRAR LAS MÁSCARAS PARA QUE SOLO QUEDEN LAS VIAS
area_minima = 10  # Valor en las mismas unidades que el CRS del shapefile

# Configurar carpeta de salida
vias_fol = mask_fol.replace('\\MASKS', '\\MASKS_VIAS')
os.makedirs(vias_fol, exist_ok=True)

# Iterar sobre los shapefiles
for shp in tqdm(list_shp):
    # Ruta del archivo de salida
    shp_out = shp.replace('\\MASKS', '\\MASKS_VIAS')
    geojson_out = shp_out.replace('.shp', '.geojson')
    
    # Verificar si el shapefile ya fue procesado
    if not os.path.exists(shp_out):
        # Leer el shapefile
        gdf = gpd.read_file(shp)
        gdf["area"] = gdf.geometry.area

        # Filtrar las geometrías cuyo área sea mayor al umbral
        gdf_filtrado = gdf[gdf["area"] > area_minima]

        # Guardar el shapefile filtrado
        gdf_filtrado.to_file(shp_out)
        gdf_filtrado.to_file(geojson_out, driver="GeoJSON")
    else:
        # Si el archivo ya existe, saltar al siguiente
        print(f"El archivo ya existe, saltando: {shp_out}")

100%|██████████| 6684/6684 [16:07<00:00,  6.91it/s]  


# **Postprocesamiento 2. Reconstrucción de polígonos continuos para dar continuidad a vías segmentadas**

Este script realiza una etapa de **postprocesamiento geométrico** sobre los shapefiles de vías (`MASKS_VIAS_`) con el objetivo de **rellenar gaps o huecos** que aparecen en las vías segmentadas. Estos espacios vacíos suelen originarse por la presencia de elementos en las imágenes tipo *Terrain* (como árboles, sombras, textos o estructuras) que interrumpen visualmente la continuidad de la vía durante el proceso de segmentación.

### 🎯 Objetivo principal:
Unificar en un **solo polígono continuo** las partes dispersas de una vía que han quedado separadas por errores visuales o ruido en la imagen, asegurando así su representación geométrica completa.

### ✅ Funcionalidades principales:

- 📏 **Filtrado adicional por área**: Se eliminan fragmentos pequeños que no aportan a la continuidad de la vía (`umbral_area`, configurable).
- 🧩 **Extracción de bordes exteriores**: Se recorre cada geometría válida y se extraen sus contornos principales.
- 🧵 **Reconstrucción de un polígono único**: A partir de todos los puntos frontera, se crea un nuevo polígono que agrupa las partes separadas en una única entidad continua.
- 💾 **Exportación organizada**: El nuevo polígono se guarda como `Shapefile` en una carpeta aparte (`MASKS_VIAS_FILL_`) para evitar sobrescritura de los archivos originales.

### 🧠 Justificación del proceso:

Durante la segmentación automática sobre imágenes satelitales del tipo **Terrain**, es común que **objetos como sombras, textos o elementos visuales** interrumpan temporalmente la detección de la vía, dividiéndola en partes aisladas. Este proceso permite **reconstruir la vía como una única entidad continua**, cerrando los huecos y generando una geometría más limpia y fiel al trazado real.

### 🛠️ Parámetros clave:

- `umbral_area`: área mínima para considerar un polígono como parte de la vía.
- `MASKS_VIAS_`: carpeta de entrada con shapefiles ya filtrados.
- `MASKS_VIAS_FILL_`: carpeta de salida con las vías reconstruidas y unificadas.

Este paso mejora la calidad y continuidad de las vías detectadas, siendo especialmente útil para análisis espaciales de conectividad, visualización precisa y entrenamiento de modelos que requieren geometrías completas.


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon
import os,glob


troncales=[#'Troncal5','Troncal2',
           'Troncal3','Troncal4','Troncal1','Troncal9']

root='./data'#Carpeta con las vías segmentadas sin postprocesamiento

In [ ]:
for troncal in troncales:
    fol=os.path.join(root,troncal)
    os.makedirs(fol.replace('MASKS_VIAS_','MASKS_VIAS_FILL_'),exist_ok=True)#Se crea una carpeta fill con el primer postprocesamiento
    print(fol)
    list_shp=glob.glob(os.path.join(fol,'*.shp'))
    print(len(list_shp))
    for shapefile_path in list_shp:
        gdf = gpd.read_file(shapefile_path)

        # Calcular el área de cada polígono
        gdf['area'] = gdf.geometry.area

        # Filtrar los polígonos con un área mayor a un umbral (por ejemplo, 1000 unidades)
        umbral_area = 1000
        gdf_filtrado = gdf[gdf['area'] > umbral_area]
        
        
        # Extraer coordenadas de los polígonos
        coords_list = []
        for geom in gdf_filtrado.geometry:
            if geom.geom_type == 'Polygon':
                coords_list.extend(geom.exterior.coords)  # Extrae solo el contorno exterior
            elif geom.geom_type == 'MultiPolygon':  # Si hay multipolígonos, extrae cada uno
                for poly in geom.geoms:
                    coords_list.extend(poly.exterior.coords)

        # Crear un nuevo polígono continuo
        if len(coords_list) > 2:  # Se necesitan al menos 3 puntos para un polígono
            nuevo_poligono = Polygon(coords_list)
            gdf_reconstruido = gpd.GeoDataFrame(geometry=[nuevo_poligono], crs=gdf_filtrado.crs)

            
            
        # Extraer coordenadas de los polígonos
        coords_list = []
        for geom in gdf_filtrado.geometry:
            if geom.geom_type == 'Polygon':
                coords_list.extend(geom.exterior.coords)  # Extrae solo el contorno exterior
            elif geom.geom_type == 'MultiPolygon':  # Si hay multipolígonos, extrae cada uno
                for poly in geom.geoms:
                    coords_list.extend(poly.exterior.coords)

        # Crear un nuevo polígono continuo
        if len(coords_list) > 2:  # Se necesitan al menos 3 puntos para un polígono
            nuevo_poligono = Polygon(coords_list)
            gdf_reconstruido = gpd.GeoDataFrame(geometry=[nuevo_poligono], crs=gdf_filtrado.crs)

            out_shp=shapefile_path.replace('MASKS_VIAS_','MASKS_VIAS_FILL_')
            gdf_reconstruido.to_file(out_shp)